In [1]:
# Data Cleaning for Recipe Dataset
import pandas as pd
import numpy as np
import re
import os
import kagglehub as kagglehub

from difflib import SequenceMatcher
from collections import Counter


/Users/makabaka/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Download dataset files to a local folder
path = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews")

print("Dataset downloaded to:", path)
print("Files:", os.listdir(path))

# Load CSVs
recipes = pd.read_csv(os.path.join(path, "recipes.csv"))
reviews = pd.read_csv(os.path.join(path, "reviews.csv"))

print(reviews.head())
print(recipes.shape, reviews.shape)

Dataset downloaded to: /Users/makabaka/.cache/kagglehub/datasets/irkaal/foodcom-recipes-and-reviews/versions/2
Files: ['reviews.csv', 'recipes.csv', 'recipes.parquet', 'reviews.parquet']
   ReviewId  RecipeId  AuthorId        AuthorName  Rating  \
0         2       992      2008         gayg msft       5   
1         7      4384      1634     Bill Hilbrich       4   
2         9      4523      2046  Gay Gilmore ckpt       2   
3        13      7435      1773     Malarkey Test       5   
4        14        44      2085        Tony Small       5   

                                              Review         DateSubmitted  \
0       better than any you can get at a restaurant!  2000-01-25T21:44:00Z   
1  I cut back on the mayo, and made up the differ...  2001-10-17T16:49:59Z   
2  i think i did something wrong because i could ...  2000-02-25T09:00:00Z   
3  easily the best i have ever had.  juicy flavor...  2000-03-13T21:15:00Z   
4                                 An excellent dish.  20

In [4]:
reviews.head()

,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25T21:44:00Z,2000-01-25T21:44:00Z
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the differ...",2001-10-17T16:49:59Z,2001-10-17T16:49:59Z
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could ...,2000-02-25T09:00:00Z,2000-02-25T09:00:00Z
3,13,7435,1773,Malarkey Test,5,easily the best i have ever had. juicy flavor...,2000-03-13T21:15:00Z,2000-03-13T21:15:00Z
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28T12:51:00Z,2000-03-28T12:51:00Z


In [6]:
reviews.columns = [
    "ReviewId", "RecipeId", "UserId", "UserName",
    "Rating", "ReviewText", "DateSubmitted", "DateModified"
]

#convert date columns to datetime format
reviews["DateSubmitted"] = pd.to_datetime(reviews["DateSubmitted"], errors="coerce")
reviews["DateModified"] = pd.to_datetime(reviews["DateModified"], errors="coerce")


In [7]:
#review dataset summary
def dataset_summary(df):
    
    def safe_nunique(col):
        try:
            return col.nunique()
        except TypeError:
            return col.astype(str).nunique()
    
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isnull().sum(),
        "missing_pct": (df.isnull().sum() / len(df) * 100).round(2),
        "n_unique": df.apply(safe_nunique)
    }).sort_values(by="missing_count", ascending=False)

    return summary

In [8]:
dataset_summary(reviews)

,dtype,missing_count,missing_pct,n_unique
ReviewText,object,214,0.02,1392745
ReviewId,int64,0,0.00,1401982
RecipeId,int64,0,0.00,271678
UserId,int64,0,0.00,271907
UserName,object,0,0.00,241365
Rating,int64,0,0.00,6
DateSubmitted,"datetime64[ns, UTC]",0,0.00,1384268
DateModified,"datetime64[ns, UTC]",0,0.00,1384268


In [12]:
#exclude rows with no reviewtext and no rating
reviews_cleaned = reviews.dropna(subset=["ReviewText"], how="all")
len(reviews) - len(reviews_cleaned)


214

In [13]:
reviews_cleaned["Rating"] = pd.to_numeric(reviews_cleaned["Rating"], errors="coerce")
dataset_summary(reviews_cleaned)

/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_44193/3741980755.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_cleaned["Rating"] = pd.to_numeric(reviews_cleaned["Rating"], errors="coerce")


,dtype,missing_count,missing_pct,n_unique
ReviewId,int64,0,0.0,1401768
RecipeId,int64,0,0.0,271670
UserId,int64,0,0.0,271725
UserName,object,0,0.0,241257
Rating,int64,0,0.0,6
ReviewText,object,0,0.0,1392745
DateSubmitted,"datetime64[ns, UTC]",0,0.0,1384054
DateModified,"datetime64[ns, UTC]",0,0.0,1384054


In [15]:
reviews_cleaned["ReviewLength"] = reviews_cleaned["ReviewText"].apply(len)

/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_44193/1263704498.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_cleaned["ReviewLength"] = reviews_cleaned["ReviewText"].apply(len)


In [16]:
reviews_cleaned["ReviewYear"] = reviews_cleaned["DateSubmitted"].dt.year
reviews_cleaned["ReviewMonth"] = reviews_cleaned["DateSubmitted"].dt.month

/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_44193/2239022390.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_cleaned["ReviewYear"] = reviews_cleaned["DateSubmitted"].dt.year
/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_44193/2239022390.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_cleaned["ReviewMonth"] = reviews_cleaned["DateSubmitted"].dt.month


In [22]:
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/makabaka/nltk_data...


In [23]:
reviews_cleaned["sentiment_score"] = reviews_cleaned["ReviewText"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_44193/4287854137.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_cleaned["sentiment_score"] = reviews_cleaned["ReviewText"].apply(


In [25]:
reviews_cleaned["sentiment_score"] = reviews_cleaned["ReviewText"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_44193/4287854137.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_cleaned["sentiment_score"] = reviews_cleaned["ReviewText"].apply(


In [27]:
review_features = reviews_cleaned.groupby("RecipeId").agg({
    "sentiment_score": "mean",
    "Rating": ["mean", "count"]
}).reset_index()

review_features.columns = [
    "RecipeId",
    "avg_sentiment",
    "avg_rating",
    "review_count"
]

In [28]:
review_features

,RecipeId,avg_sentiment,avg_rating,review_count
0,38,0.709500,4.250000,4
1,39,0.877800,3.000000,1
2,40,0.754033,4.333333,9
3,41,0.928200,4.500000,2
4,42,0.668922,2.666667,9
...,...,...,...,...
271665,540899,0.688400,5.000000,1
271666,541001,0.690800,0.000000,1
271667,541030,0.690800,5.000000,1
271668,541195,0.907800,5.000000,1


In [29]:
from sklearn.preprocessing import StandardScaler

reviews_cleaned["Rating_scaled"] = StandardScaler().fit_transform(reviews_cleaned[["Rating"]])

/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_44193/2299245587.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_cleaned["Rating_scaled"] = StandardScaler().fit_transform(reviews_cleaned[["Rating"]])


In [30]:
reviews_cleaned.head()


,ReviewId,RecipeId,UserId,UserName,Rating,ReviewText,DateSubmitted,DateModified,ReviewLength,ReviewYear,ReviewMonth,sentiment_score,Rating_scaled
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25 21:44:00+00:00,2000-01-25 21:44:00+00:00,44,2000,1,0.4926,0.465457
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the differ...",2001-10-17 16:49:59+00:00,2001-10-17 16:49:59+00:00,102,2001,10,-0.2732,-0.320661
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could ...,2000-02-25 09:00:00+00:00,2000-02-25 09:00:00+00:00,91,2000,2,-0.4767,-1.892897
3,13,7435,1773,Malarkey Test,5,easily the best i have ever had. juicy flavor...,2000-03-13 21:15:00+00:00,2000-03-13 21:15:00+00:00,119,2000,3,0.8398,0.465457
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28 12:51:00+00:00,2000-03-28 12:51:00+00:00,18,2000,3,0.5719,0.465457


In [32]:
reviews_cleaned.head(500).to_csv("reviews_cleaned_sample500.csv", index=False)